In [1]:
import kaleidocell
import scanpy as sc
import os


In [2]:
adata = sc.read_h5ad("/net/bq-storage/ag-cherrmann/bq_cinac/projects/synimmune/data/01_extGBmap/immune_subsets/Mono.h5ad")


In [3]:
print(adata.obs.columns.tolist())
print(adata.obs["donor_id"].nunique())
print(f"{adata.obs['annotation_level_3'].unique()[0]}: {adata.n_obs} cells")

['author', 'donor_id', 'assay_ontology_term_id', 'cell_type_ontology_term_id', 'development_stage_ontology_term_id', 'disease_ontology_term_id', 'self_reported_ethnicity_ontology_term_id', 'is_primary_data', 'sex_ontology_term_id', 'annotation_level_1', 'annotation_level_2', 'annotation_level_3', 'gbmap', 'suspension_type', 'stage', 'location', 'sector', 'celltype_original', 'EGFR', 'MET', 'p53', 'TERT', 'ATRX', 'PTEN', 'MGMT', 'chr1p19q', 'PDGFR', 'tissue_ontology_term_id', 'tissue_type', 'cell_type', 'assay', 'disease', 'sex', 'tissue', 'self_reported_ethnicity', 'development_stage', 'observation_joinid']
189
Mono: 41018 cells


In [4]:
print(adata.obs["donor_id"].value_counts().describe())
print(adata.obs["donor_id"].value_counts())

count     189.000000
mean      217.026455
std       673.948870
min         1.000000
25%         7.000000
50%        27.000000
75%       104.000000
max      5773.000000
Name: count, dtype: float64
donor_id
rGBM-02                5773
R5                     5110
Patient_1              3397
Patient_4              2616
BT397                  1543
                       ... 
PJ048                     1
SF11956                   1
SF11979                   1
Pat9_Friedrich2021        1
Pat14_Friedrich2021       1
Name: count, Length: 189, dtype: int64


In [4]:
results_nmf, nmf_convergence = kaleidocell.multi_sample_nmf(
    adata,
    batch_key="donor_id",
    test_ranks=[4, 5, 6, 7, 8, 9],
    n_initializations=1,
    max_iterations=100
)

Running multi-sample NMF on 189 samples.
Tested ranks: [4, 5, 6, 7, 8, 9]


Global NMF Progress:   0%|          | 0/113400 [00:00<?, ?it/s]

/net/bq-storage/ag-cherrmann/bq_cinac/projects/synimmune/.environments/kaleidocell_env/lib/python3.11/site-packages/kaleidocell/nmf.py:104: UserWarning: Number of rows/columns is smaller than rank; results may be unstable.
  warnings.warn(
/net/bq-storage/ag-cherrmann/bq_cinac/projects/synimmune/.environments/kaleidocell_env/lib/python3.11/site-packages/kaleidocell/nmf.py:104: UserWarning: Number of rows/columns is smaller than rank; results may be unstable.
  warnings.warn(
/net/bq-storage/ag-cherrmann/bq_cinac/projects/synimmune/.environments/kaleidocell_env/lib/python3.11/site-packages/kaleidocell/nmf.py:104: UserWarning: Number of rows/columns is smaller than rank; results may be unstable.
  warnings.warn(
/net/bq-storage/ag-cherrmann/bq_cinac/projects/synimmune/.environments/kaleidocell_env/lib/python3.11/site-packages/kaleidocell/nmf.py:104: UserWarning: Number of rows/columns is smaller than rank; results may be unstable.
  warnings.warn(
/net/bq-storage/ag-cherrmann/bq_cinac/pr

Multi-sample NMF complete.


In [8]:
results_mp = kaleidocell.derive_nmf_metaprograms(results_nmf)
print(results_mp["metrics"])

Total number of programs: 7371
Optimal number of meta-programs: 30
Clustering using hierarchical Ward linkage
Defining meta-program consensus
Computing meta-program metrics
Meta-program derivation complete.
      sampleCoverage  silhouette  meanSimilarity  nPrograms  nGenes
MP1         0.211640    0.087026           0.265        182      23
MP2         0.269841    0.184803           0.339        226      10
MP3         0.227513    0.213482           0.326        181       3
MP4         0.211640    0.103064           0.273        171      10
MP5         0.105820    0.203943           0.355         85      10
MP6         0.291005   -0.061105           0.123        245     116
MP7         0.137566    0.332360           0.396         61       7
MP8         0.047619    0.284514           0.351         44      10
MP9         0.169312   -0.014627           0.134        140     630
MP10        0.195767    0.020836           0.165        124       3
MP11        0.010582    0.292910           0.

In [7]:
mp_scores = kaleidocell.compute_mp_scores(results_mp, adata)

Computing module scores per MP.
Finished computing MP scores.


In [9]:
path = kaleidocell.get_html(
    results_mp,
    adata,
    mp_scores=mp_scores,
    obs=["annotation_level_3", "stage", "donor_id", "disease"],
    output_path="/net/bq-storage/ag-cherrmann/bq_cinac/projects/synimmune/results/02_NMF_immune_subsets/02_nmf_Mono"
)
print(f"Report written to: {path}")

Rendering heatmap…
Rendering UMAP scores…
Building metrics table…
Running GSEA — GO Biological Process…
Ensembl gene IDs detected — translating to HGNC using bundled table.
Building gene table…
Rendering violin plots for 'annotation_level_3'…
Plotting MP distributions across 'annotation_level_3'.


/net/bq-storage/ag-cherrmann/bq_cinac/projects/synimmune/.environments/kaleidocell_env/lib/python3.11/site-packages/kaleidocell/visualization.py:360: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.violinplot(data=df, x=batch_key, y=score_col, palette=palette, inner="box")
/net/bq-storage/ag-cherrmann/bq_cinac/projects/synimmune/.environments/kaleidocell_env/lib/python3.11/site-packages/kaleidocell/visualization.py:360: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.violinplot(data=df, x=batch_key, y=score_col, palette=palette, inner="box")
/net/bq-storage/ag-cherrmann/bq_cinac/projects/synimmune/.environments/kaleidocell_env/lib/python3.11/site-packages/kaleidocell/visualization.py:360: FutureWarning: 

Pas

Finished plotting MP distributions.
Rendering violin plots for 'stage'…
Plotting MP distributions across 'stage'.


/net/bq-storage/ag-cherrmann/bq_cinac/projects/synimmune/.environments/kaleidocell_env/lib/python3.11/site-packages/kaleidocell/visualization.py:360: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.violinplot(data=df, x=batch_key, y=score_col, palette=palette, inner="box")
/net/bq-storage/ag-cherrmann/bq_cinac/projects/synimmune/.environments/kaleidocell_env/lib/python3.11/site-packages/kaleidocell/visualization.py:360: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.violinplot(data=df, x=batch_key, y=score_col, palette=palette, inner="box")
/net/bq-storage/ag-cherrmann/bq_cinac/projects/synimmune/.environments/kaleidocell_env/lib/python3.11/site-packages/kaleidocell/visualization.py:360: FutureWarning: 

Pas

Finished plotting MP distributions.
Rendering violin plots for 'donor_id'…
Plotting MP distributions across 'donor_id'.


/net/bq-storage/ag-cherrmann/bq_cinac/projects/synimmune/.environments/kaleidocell_env/lib/python3.11/site-packages/kaleidocell/visualization.py:360: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.violinplot(data=df, x=batch_key, y=score_col, palette=palette, inner="box")
/net/bq-storage/ag-cherrmann/bq_cinac/projects/synimmune/.environments/kaleidocell_env/lib/python3.11/site-packages/kaleidocell/visualization.py:360: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.violinplot(data=df, x=batch_key, y=score_col, palette=palette, inner="box")
/net/bq-storage/ag-cherrmann/bq_cinac/projects/synimmune/.environments/kaleidocell_env/lib/python3.11/site-packages/kaleidocell/visualization.py:360: FutureWarning: 

Pas

Finished plotting MP distributions.
Rendering violin plots for 'disease'…
Plotting MP distributions across 'disease'.


/net/bq-storage/ag-cherrmann/bq_cinac/projects/synimmune/.environments/kaleidocell_env/lib/python3.11/site-packages/kaleidocell/visualization.py:360: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.violinplot(data=df, x=batch_key, y=score_col, palette=palette, inner="box")
/net/bq-storage/ag-cherrmann/bq_cinac/projects/synimmune/.environments/kaleidocell_env/lib/python3.11/site-packages/kaleidocell/visualization.py:360: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.violinplot(data=df, x=batch_key, y=score_col, palette=palette, inner="box")
/net/bq-storage/ag-cherrmann/bq_cinac/projects/synimmune/.environments/kaleidocell_env/lib/python3.11/site-packages/kaleidocell/visualization.py:360: FutureWarning: 

Pas

Finished plotting MP distributions.

Files written to /net/bq-storage/ag-cherrmann/bq_cinac/projects/synimmune/results/02_NMF_immune_subsets/02_nmf_Mono/
  results.html
  gsea_GO_Biological_Process.csv
  genes.csv
Report written to: /net/bq-storage/ag-cherrmann/bq_cinac/projects/synimmune/results/02_NMF_immune_subsets/02_nmf_Mono/results.html
